In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from prophet import Prophet
from prophet.plot import plot_plotly

/Users/lorenzodimaio/Documents/Iot_project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1. Setup and Data Cleaning

In [ ]:
# 1. Load the dataset
df = pd.read_parquet('archive/dataset_vigna.parquet')

# 1. Convert timestamp to datetime format
df['timestamp'] = pd.to_datetime(df['timestamp'])

# 2. Sort data chronologically (for time series)
df = df.sort_values('timestamp').reset_index(rop=True)

# 3. Remove duplicates
df = df.drop_duplicates(subset=['timestamp', 'sensor_id'])

# 4. Clean negative values (Except temperature)
numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
exceptions = ['temperature', 'weather_temp']
to_check = [c for c in numeric_columns if c not in exceptions]
df = df[(df[to_check] >= 0).all(axis=1)]

# 5. Interpolation (Filling data gaps)
df['moisture'] = df['moisture'].interpolate()
df['temperature'] = df['temperature'].interpolate()


print(f"Total rows in dataset: {len(df)}")
df.head(5)

Total rows in dataset: 314


,id,sensor_id,timestamp,temperature,humidity,moisture,grape_count,health_status,estimated_liters,leaf_healthy_count,leaf_stress_count,leaf_disease_count,sector_id,external_id,weather_temp,weather_humidity,weather_rain
0,1617,272,2026-04-21 13:00:00,27.50,49.02,43.83,15.0,Healthy,0.20,96,1,0,1,S-01,18.18,54.403301,0.9
1,1618,273,2026-04-21 13:00:00,28.71,47.63,25.60,20.0,Healthy,0.41,87,1,0,2,S-02,18.18,54.403301,0.9
2,1620,275,2026-04-21 13:00:00,27.62,50.34,26.99,30.0,Healthy,0.59,81,0,0,4,S-04,18.18,54.403301,0.9
3,1621,276,2026-04-21 13:00:00,28.67,46.48,30.77,23.0,Healthy,0.21,87,3,0,5,S-05,18.18,54.403301,0.9
4,1622,277,2026-04-21 13:00:00,28.19,49.80,36.16,19.0,Healthy,0.27,93,4,1,6,S-06,18.18,54.403301,0.9


# 2. Create Global DF

In [3]:
global_df = df.groupby('timestamp').agg({
    'moisture': 'mean',
    'temperature': 'mean',
    'weather_rain': 'mean',
    'grape_count': 'sum',
    'humidity': 'mean'
}).reset_index()
global_df.head()

,timestamp,moisture,temperature,weather_rain,grape_count,humidity
0,2026-04-21 13:00:00,33.815000,27.880000,0.9,160.0,48.306250
1,2026-04-21 14:00:00,36.671667,27.860000,0.5,132.0,47.560000
2,2026-04-21 15:00:00,37.752000,28.082000,0.1,107.0,48.754000
3,2026-04-21 16:00:00,35.976667,27.508333,0.0,126.0,49.846667
4,2026-04-21 17:00:00,38.215000,26.215000,0.0,99.0,52.700000


# 3. Prophet Model for Moisture

In [4]:
# Prepare the dataset for Prophet with Regressors
moisture_prophet_df = global_df[['timestamp', 'moisture', 'temperature', 'weather_rain']].copy()
moisture_prophet_df.columns = ['ds', 'y', 'temp', 'rain']
moisture_prophet_df['ds'] = moisture_prophet_df['ds'].dt.tz_localize(None)
# Initialize the model with Extra Regressors
m = Prophet(daily_seasonality=True, weekly_seasonality=True)
m.add_regressor('temp')
m.add_regressor('rain')
# Training
m.fit(moisture_prophet_df)

12:06:48 - cmdstanpy - INFO - Chain [1] start processing
12:06:48 - cmdstanpy - INFO - Chain [1] done processing


In [5]:
# Create future dates for the next 3 days (72 hours)
future = m.make_future_dataframe(periods=72, freq='h')
# Weather simulation for the future (e.g., 28 stable degrees and 0 rain)
future['temp'] = moisture_prophet_df['temp'].tolist() + [28.0] * 72
future['rain'] = moisture_prophet_df['rain'].tolist() + [0.0] * 72
# Prediction
moisture_humidity_forecast = m.predict(future)

fig = plot_plotly(m, moisture_humidity_forecast)
fig.update_layout(
    title="Prediction: Soil Moisture (Next 72h)",
    template="plotly_dark",
    xaxis_title="Time",
    yaxis_title="Moisture (%)"
)
fig.show()

# 4. Prophet Model for Temperature

In [10]:
# 1. Prepare data
frost_prophet_df = global_df[['timestamp', 'temperature', 'weather_rain', 'humidity']].copy()
frost_prophet_df.columns = ['ds', 'y', 'rain', 'air_humidity']
frost_prophet_df['ds'] = frost_prophet_df['ds'].dt.tz_localize(None)

# 2. Initialize model for Temperature
m_frost = Prophet(daily_seasonality=True, weekly_seasonality=False)
# Add air humidity as a regressor (it strongly affects night frost)
m_frost.add_regressor('air_humidity')
m_frost.add_regressor('rain')
m_frost.fit(frost_prophet_df)
# 3. Create future dates for the next 48 hours (two nights)
future_frost = m_frost.make_future_dataframe(periods=48, freq='h')
# Simulate future air humidity (e.g., 60%) and 0 rain
future_frost['air_humidity'] = frost_prophet_df['air_humidity'].tolist() + [60.0] * 48
future_frost['rain'] = frost_prophet_df['rain'].tolist() + [0.0] * 48
# 4. Prediction
temperature_forecast = m_frost.predict(future_frost)


fig = plot_plotly(m_frost, temperature_forecast)
fig.update_layout(
    title="Prediction: Temperature (Next 48h)",
    template="plotly_dark",
    xaxis_title="Time",
    yaxis_title="Temperature (°C)"
)
fig.show()

12:18:54 - cmdstanpy - INFO - Chain [1] start processing
12:18:54 - cmdstanpy - INFO - Chain [1] done processing


# 5. Alert Logic

## 5.1. Frost Alert

In [11]:
FROST_THRESHOLD = 2.0

# 1. Analyze the next 48 hours
frost_future_data = temperature_forecast.tail(48)

# 2. Find the absolute cold peak
frost_min_index = frost_future_data['yhat'].idxmin()
min_temp = frost_future_data.loc[frost_min_index, 'yhat']
min_time = frost_future_data.loc[frost_min_index, 'ds']

# 3. Find when temperature drops below the safety threshold
frost_below_threshold = frost_future_data[frost_future_data['yhat'] <= FROST_THRESHOLD]

print("-" * 45)
if not frost_below_threshold.empty:
    danger_start_time_frost = frost_below_threshold.iloc[0]['ds']
    
    print(f"❄️ FROST ALERT: Thermal risk detected!")
    print(f"⏳ Alert start: {danger_start_time_frost} (Drops below {FROST_THRESHOLD}°C)")
    print(f"📉 Cold peak: {min_time} (Value: {min_temp:.2f}°C)")
    print("-" * 45)
    print("Advice: Activate protection systems 30 minutes before the alert start.")
else:
    print(f"✅ NO FROST RISK")
    print(f"🔝 Min predicted temperature: {min_temp:.2f}°C at {min_time}")
print("-" * 45)


---------------------------------------------
✅ NO FROST RISK
🔝 Min predicted temperature: 12.84°C at 2026-04-24 02:00:00
---------------------------------------------


## 5.2. Moisture Alert

In [9]:
DANGER_THRESHOLD = 25.0

# 1. Find the absolute MINIMUM point
min_index = moisture_humidity_forecast['yhat'].tail(72).idxmin()
min_value = moisture_humidity_forecast.loc[min_index, 'yhat']
min_time = moisture_humidity_forecast.loc[min_index, 'ds']

# 2. Find when the problem STARTS (the first time it drops below the threshold)
future_data = moisture_humidity_forecast.tail(72)
below_threshold = future_data[future_data['yhat'] < DANGER_THRESHOLD]

print("-" * 45)
if not below_threshold.empty:
    danger_start_time = below_threshold.iloc[0]['ds']
    
    print(f"🚨 ALERT: Water stress period detected!")
    print(f"📅 Criticality start: {danger_start_time} (Drops below {DANGER_THRESHOLD}%)")
    print(f"📉 Worst point:   {min_time} (Value: {min_value:.2f}%)")
else:
    print(f"✅ ALL OK: Moisture will remain above the safety threshold.")
    print(f"🔝 Min predicted value: {min_value:.2f}% at {min_time}")
print("-" * 45)


---------------------------------------------
🚨 ALERT: Water stress period detected!
📅 Criticality start: 2026-04-24 04:00:00 (Drops below 25.0%)
📉 Worst point:   2026-04-25 15:00:00 (Value: -219.22%)
---------------------------------------------
